# Top blades — render the 3D-verified winners, and why they're all cups

Fresh session, **re-runs no CFD**. Loads the campaign's `pareto.json` and (if present) the
Phase-5 `verification.json`, ranks the **valid** designs by their 3D-verified `J_fan`, and draws
each as a **draggable** Plotly solid at *true* proportions (drag to rotate, scroll to zoom). Then
it decomposes every blade's shape into **rib bow (cup)** vs **panel displacement (zigzag/camber)**
to show *why* the optimizer produced cups and no zigzags. CadQuery + Plotly only — no gmsh/SU2.

## 1. Repo + deps (+ Drive on Colab)

In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "blade-redesign-aero-first"
if IN_COLAB:
    REPO = Path("/content/fan-optimization")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cadquery", "plotly"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    REPO = Path.cwd()
    while REPO != REPO.parent and not (REPO / "pyproject.toml").exists():
        REPO = REPO.parent
    DATA_ROOT = REPO / "data"
for p in (str(REPO), str(REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| colab:", IN_COLAB, "| data root:", DATA_ROOT)

## 2. Imports\n\nAll project + plotting imports in one cell.

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go

from fanopt.geometry.blade import BladeParams
from fanopt.geometry.blade_cad import blade_trimesh
from fanopt.utils.ledger import design_hash

## 3. Load the Pareto set + 3D verification (if present)

`pareto.json` is the campaign output (every non-dominated design). `verification.json` (Phase-5)
adds each top design's **3D** `J_fan` and a `suspect` flag (failed / reverse-thrust). We rank the
**valid** designs by 3D `J_fan`; if verification isn't in yet, we fall back to the 2D-slice
`J_fan` and say so.

In [ ]:
CAMPAIGN = "phase4_aero_v2"   # set to your campaign OUT_DIR name (_v2 = enriched 22-var meridian)
PARETO = DATA_ROOT / CAMPAIGN / "pareto.json"
VERIFY = DATA_ROOT / "phase5_verify_blade" / "verification.json"

pareto = json.loads(PARETO.read_text())
by_hash = {design_hash(d["params"]): d for d in pareto}

if VERIFY.exists():
    vj = json.loads(VERIFY.read_text())
    suspect_of = {p["name"]: p["suspect"] for p in vj["ranking"]["pairs"]}
    ranked = []
    for dz in vj["designs"]:
        h = dz["name"].split("_", 1)[1]            # name = "<rank>_<hash>"
        d = by_hash.get(h)
        if d is None:
            continue
        ranked.append({**d, "j_fan_3d": dz["j_fan_3d"], "suspect": suspect_of.get(dz["name"], False)})
    valid = [r for r in ranked if not r["suspect"] and r["j_fan_3d"] is not None]
    valid.sort(key=lambda r: r["j_fan_3d"], reverse=True)
    RANK_LABEL = "3D-verified J_fan"
    print(f"{len(ranked)} verified, {len(valid)} valid  ->  ranking by {RANK_LABEL}")
else:
    valid = sorted(pareto, key=lambda d: d["j_fan"], reverse=True)
    for r in valid:
        r["j_fan_3d"] = None
    RANK_LABEL = "2D-slice J_fan (no verification.json yet - run Phase-5 verify for 3D)"
    print(f"no verification.json - ranking {len(valid)} Pareto designs by {RANK_LABEL}")

## 4. Draggable render helper (true proportions)

`params -> make_blade_solid -> tessellate -> Plotly Mesh3d`. `aspectmode='data'` keeps the real
shape (a shallow cup) instead of stretching it to a cube.

In [ ]:
def blade_figure(d, title):
    V, F = blade_trimesh(BladeParams.from_dict(d["params"]))
    mesh = go.Mesh3d(x=V[:, 0] * 1000, y=V[:, 1] * 1000, z=V[:, 2] * 1000,
                     i=F[:, 0], j=F[:, 1], k=F[:, 2],
                     color="seagreen", opacity=0.7, flatshading=True)
    fig = go.Figure(data=[mesh])
    fig.update_layout(title=title, height=520, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data", xaxis_title="x (mm)",
                                 yaxis_title="y (mm)", zaxis_title="z (mm)"))
    return fig

## 5. The top 5 valid blades (drag to rotate)

The highest-`J_fan` designs that passed 3D verification — your real candidates. Rotate each to
see the cup depth; at true proportions they are shallow dished sectors, not tents.

In [ ]:
TOP_N = 5
for rank, d in enumerate(valid[:TOP_N], 1):
    j3d = d.get("j_fan_3d")
    tag = f"3D={j3d:.2e}" if j3d is not None else f"slice={d['j_fan']:.2e}"
    p = d["params"]
    blade_figure(d, f"#{rank}  {tag}  |  {d['mass_kg'] * 1000:.0f} g / {p['blade_count']} blades").show()

## 6. Why are they all cups — and where did the zigzag go?

Early on, a free-form panel *zigzag* looked like it made more wind, yet the optimizer found none.
This is **not** a preference — the parameterization can barely *express* a zigzag. A blade's
out-of-plane shape has two sources:

- **Rib bow** — the `)`-shaped meridian revolved about the pin (`rib_bow_knots_m`, **0–30 mm** per
  knot). A surface of revolution **nests when folded**, so it's unconstrained → cups *and* radial
  pleats/zigzag live here (that's the enriched meridian).
- **Panel displacement** — the free grid riding on the rib surface. But to **fold flat**, the panel
  must stay inside the rib's thickness envelope `(t_rib − panel)/2`, which is **sub-millimetre**
  (thin ribs to save mass, thin panel). So any zigzag is capped at a fraction of a mm — invisible
  next to a 20 mm cup.

The table + chart below show it per design: bow in **mm**, panel in **mm**, and their ratio.

In [ ]:
def bow_mm(p):  # works for both schemas (from_dict resamples the legacy 2-knot meridian)
    return max(abs(k) for k in BladeParams.from_dict(p).rib_bow_knots_m) * 1000

def panel_mm(p):
    return max(abs(v) for row in p["panel_offsets_m"] for v in row) * 1000

print(f"{'rank':>4} {'J_fan':>11} {'bow(mm)':>8} {'panel(mm)':>10} {'bow/panel':>10}")
for rank, d in enumerate(valid[:10], 1):
    p = d["params"]; b = bow_mm(p); pn = panel_mm(p)
    jf = d.get("j_fan_3d") or d["j_fan"]
    print(f"{rank:>4} {jf:11.2e} {b:8.1f} {pn:10.3f} {b / max(pn, 1e-9):10.0f}")

In [ ]:
n = min(10, len(valid))
xs = [f"#{i+1}" for i in range(n)]
fig = go.Figure()
fig.add_bar(x=xs, y=[bow_mm(valid[i]["params"]) for i in range(n)], name="rib bow (cup)")
fig.add_bar(x=xs, y=[panel_mm(valid[i]["params"]) for i in range(n)], name="panel offset (zigzag)")
fig.update_layout(barmode="group", yaxis_type="log", yaxis_title="amplitude (mm, log)",
                  xaxis_title="design (by J_fan)", height=400,
                  title="Cup vs zigzag amplitude (log) - the panel is ~100x smaller than the bow")
fig.show()

## 7. Takeaway

- The candidates are **shallow cups** because, under V1's fold-by-nesting rule, the rib meridian is
  the only large-amplitude shape DOF; the panel zigzag is throttled to sub-mm by containment.
- To let zigzags / corrugation / vents compete on aero merit you must **decouple panel shape from
  the fold envelope** — the V2 free-form direction (`docs/V2_backlog.md`), i.e. a different fold
  mechanism (or accepting thicker folded stacks).
- Once the top-10 3D verification finishes, re-run this notebook: it will rank by real 3D `J_fan`
  and the `suspect` flag drops any cup that didn't survive the 3D physics.